In [1]:
!pip install gtfparse
!pip install polars=='0.16.17'
!pip install pyarrow
!pip install anndata==0.8.0

In [ ]:
#Inputs: csv outputs of SoupX ambient RNA removal pipeline
#Ouputs: combined anndata object

In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import pyarrow
from gtfparse import read_gtf
import scipy

In [7]:
#Input all csv outputs of SoupX
dat_list = ['MO/Outputs/Run12_sample2_soupcorrected_plus5.csv',
           'MO/Outputs/Run12_sample3_soupcorrected_plus5.csv',
           'MO/Outputs/Run12_sample6_soupcorrected_plus5.csv',
           'MO/Outputs/Run12_sample7_soupcorrected_plus5.csv',
           'MO/Outputs/Run12_sample9_soupcorrected_plus5.csv',
           'MO/Outputs/Run12_sample10_soupcorrected_plus5.csv',
           'MO/Outputs/Run12_sample11_soupcorrected_plus5.csv',
           'MO/Outputs/Run12_sample12_soupcorrected_plus5.csv',]

In [ ]:
#Input the first sample here
tot_dat = ad.read_csv('MO/Outputs/Run12_sample1_soupcorrected_plus5.csv')

tot_dat = tot_dat.T
tot_dat.X = scipy.sparse.csr_matrix(tot_dat.X)
counts = np.sum(tot_dat.X, axis = 1).A.reshape((1,len(tot_dat)))[0]
genes = np.count_nonzero(tot_dat.X.A, axis = 1)

tot_dat.obs['n_counts'] = counts
tot_dat.obs['n_genes'] = genes
#manually set key name
tot_dat.obs['key'] = 'Run12_sample1'
tot_dat.var_names = [i for i in tot_dat.var_names]

for dn in dat_list:
    dat = ad.read_csv(dn)
    dat = dat.T
    dat.X = scipy.sparse.csr_matrix(dat.X)
    counts = np.sum(dat.X, axis = 1).A.reshape((1,len(dat)))[0]
    genes = np.count_nonzero(dat.X.A, axis = 1)
    
    dat.obs['n_counts'] = counts
    dat.obs['n_genes'] = genes
    #set this such that it outputs the key that you would like
    dat.obs['key'] = dn.split('_')[0].split('/')[-1] + '_' + dn.split('_')[1]
    dat.var_names = [i for i in dat.var_names]
    
    tot_dat = ad.concat([tot_dat, dat], join = 'outer')
    print(tot_dat.shape)

In [ ]:
tot_dat.var_names

In [ ]:
#input the gtf that you are using so that you can get gene names (only for prairie vole)
db_gene = read_gtf('MO/Microtus_ochrogaster.MicOch1.0.112.gtf', features = ['gene', 'transcript'])
df_gene = db_gene.to_pandas()

In [ ]:
df_gene

In [ ]:
#rename genes with gene name instead of gene ID (only for prairie vole)
fin_gene_name = {}
for item in tot_dat.var_names:
    gn = df_gene.loc[df_gene.index[df_gene['gene_id'] == item][0], 'gene_name']
    if gn != "":
        fin_gene_name[item] = gn
    else:
        fin_gene_name[item] = item

In [21]:
#check to see if gene names match names in BLAST tables
mapping = pd.read_csv('../../BLASTMAPPING/maps/active_maps/hypo_proj/mgmo/mg_to_mo.txt', delimiter = '\t', header = None)

In [ ]:
tot_dat.var_names = [fin_gene_name[i] for i in tot_dat.var_names]

In [ ]:
a = 0
mo_set = set(mapping[1].unique())
for item in tot_dat.var_names:
    if item in mo_set:
        a += 1
a

In [ ]:
tot_dat.var_names

In [25]:
tot_dat.obs_names_make_unique()
tot_dat.var_names_make_unique()

In [ ]:
tot_dat.obs['key'].unique()

In [12]:
tot_dat.write('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_MO_soupX_plus5.h5ad')